In [1]:
import os
import pandas as pd
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel
import torch.nn.functional as F

BASE_DIR = "Scientific_Novelty_Detection_2022_2025"
TRIPLET_DIR = os.path.join(BASE_DIR, "Triplets", "Blogs")

TASKS = ["Dia", "MT", "NLI", "Par", "QA", "SA", "Sum"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [2]:
model_name = "allenai/scibert_scivocab_uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()

print("SciBERT loaded.")

C:\Users\spars\AppData\Roaming\Python\Python311\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
C:\Users\spars\anaconda3\Lib\site-packages\transformers\utils\generic.py:260: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
C:\Users\spars\anaconda3\Lib\site-packages\transformers\utils\generic.py:260: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
C:\Users\spars\anaconda3\Lib\site-packages\transformers\modeling_utils.py:479: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value),

SciBERT loaded.


In [3]:
def embed_text_batch(text_list, batch_size=32):

    embeddings = []

    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i+batch_size]

        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            return_tensors="pt",
            max_length=128
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)

        batch_embeddings = outputs.last_hidden_state[:, 0, :]
        embeddings.append(batch_embeddings.cpu())

    return torch.cat(embeddings, dim=0)

In [6]:
def compute_blogs_predicate_weights(task):

    input_path = os.path.join(TRIPLET_DIR, f"{task}_triplets_results.csv")
    output_path = os.path.join(TRIPLET_DIR, f"{task}_triplets_weights.csv")

    # ==============================
    # Safety Checks
    # ==============================
    if not os.path.exists(input_path):
        print(f"{task} Blogs triplets file not found.")
        return

    if os.path.exists(output_path):
        print(f"{task} Blogs weights already computed.")
        return

    print(f"\nProcessing Blogs weights for {task}")

    df = pd.read_csv(input_path)

    if df.empty:
        print(f"{task} Blogs triplets file is empty.")
        return

    # Ensure required columns exist
    required_cols = ["sub", "pred", "obj"]
    for col in required_cols:
        if col not in df.columns:
            print(f"Missing column {col} in {task}")
            return

    predicates = df["pred"].astype(str).tolist()
    sub_obj = (df["sub"].astype(str) + " " + df["obj"].astype(str)).tolist()

    # ==============================
    # Embedding Function (Batch Safe)
    # ==============================
    def embed_text_batch(text_list, batch_size=32):

        embeddings = []

        for i in range(0, len(text_list), batch_size):
            batch = text_list[i:i+batch_size]

            inputs = tokenizer(
                batch,
                padding=True,
                truncation=True,
                return_tensors="pt",
                max_length=128
            ).to(device)

            with torch.no_grad():
                outputs = model(**inputs)

            batch_embeddings = outputs.last_hidden_state[:, 0, :]
            embeddings.append(batch_embeddings.cpu())

            # Clear GPU memory
            torch.cuda.empty_cache()

        return torch.cat(embeddings, dim=0)

    # ==============================
    # Compute Embeddings
    # ==============================
    print("Embedding predicates...")
    pred_embeddings = embed_text_batch(predicates)

    print("Embedding subject-object...")
    so_embeddings = embed_text_batch(sub_obj)

    print("Computing cosine similarity...")

    similarities = torch.nn.functional.cosine_similarity(
        pred_embeddings,
        so_embeddings,
        dim=1
    )

    df["pred_weights"] = similarities.numpy()

    # ==============================
    # Save Output
    # ==============================
    df.to_csv(output_path, index=False)

    print(f"{task} Blogs weights saved successfully.")

In [7]:
for task in TASKS:
    compute_blogs_predicate_weights(task)


Processing Blogs weights for Dia
Embedding predicates...
Embedding subject-object...
Computing cosine similarity...
Dia Blogs weights saved successfully.

Processing Blogs weights for MT
Embedding predicates...
Embedding subject-object...
Computing cosine similarity...
MT Blogs weights saved successfully.

Processing Blogs weights for NLI
Embedding predicates...
Embedding subject-object...
Computing cosine similarity...
NLI Blogs weights saved successfully.

Processing Blogs weights for Par
Embedding predicates...
Embedding subject-object...
Computing cosine similarity...
Par Blogs weights saved successfully.

Processing Blogs weights for QA
Embedding predicates...
Embedding subject-object...
Computing cosine similarity...
QA Blogs weights saved successfully.

Processing Blogs weights for SA
Embedding predicates...
Embedding subject-object...
Computing cosine similarity...
SA Blogs weights saved successfully.

Processing Blogs weights for Sum
Embedding predicates...
Embedding subject-